# 11 — Evolve and assure the contract

Observe a compatibility gate, classify an omitted-input change and record an explicit major-version review.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/11-assure-the-contract/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/11-assure-the-contract/lesson.md) · [Exercise](../modules/11-assure-the-contract/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/11-assure-the-contract"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Compare versioned interfaces

Version 1 allows callers to omit EPSG because it has a default. Version 2 removes that default. Predict whether an existing job omitting EPSG can still run, and which SemVer change is required.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
import os
import yaml
root = Path(os.environ['WORKSPACE']) / 'modules/11-assure-the-contract'
for version in ['v1', 'v2']:
    contract = yaml.safe_load((root / version / 'waterbodies.cwl').read_text())
    workflow = next(process for process in contract['$graph'] if process['class'] == 'Workflow')
    print(version, contract['s:softwareVersion'], workflow['inputs']['epsg'])
PYTHON
transpiler-mate baseline "$WORKSPACE/modules/11-assure-the-contract/v2/waterbodies.cwl"     --previous "$WORKSPACE/modules/11-assure-the-contract/v1/waterbodies.cwl" --output baseline.json
cat baseline.json
)


## Observe the unresolved review

The following gate is expected to fail until behavioral changes are reviewed. Capture that failure explicitly; a green notebook cell here means the expected rejection occurred, not that the compatibility gate accepted the release.


In [ ]:
(
set -euo pipefail
if transpiler-mate baseline "$WORKSPACE/modules/11-assure-the-contract/v2/waterbodies.cwl"     --previous "$WORKSPACE/modules/11-assure-the-contract/v1/waterbodies.cwl" --output unresolved.json --check > unresolved.log 2>&1; then
    printf 'Unexpected success without compatibility review.\n' >&2
    exit 1
fi
cat unresolved.log
)


## Record the known major-change review

After examining the findings, this example records `--review-bump major`. That is a review decision, not a blanket way to silence future compatibility findings.


In [ ]:
(
set -euo pipefail
transpiler-mate baseline "$WORKSPACE/modules/11-assure-the-contract/v2/waterbodies.cwl"     --previous "$WORKSPACE/modules/11-assure-the-contract/v1/waterbodies.cwl" --output reviewed.json --review-bump major --check
python - <<'PYTHON'
from pathlib import Path
import json
report = json.loads(Path('reviewed.json').read_text())
assert any(finding['rule'] == 'input.omission_removed' for finding in report['findings'])
assert report['minimum_version'] == '2.0.0'
assert report['review_bump'] == 'major' and not report['review_required']
print('Minimum version:', report['minimum_version'], 'Review:', report['review_bump'])
PYTHON
)


## Record your findings

**Expected outcome:** The report identifies input.omission_removed, requires at least 2.0.0 and accepts the explicitly reviewed major change.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/11-assure-the-contract/solution.md) after the activity. Return to [the lab index](README.md).
